# Atlas Visualization

This notebook covers three complementary visualization approaches:

1. **Pre-built figures** — static PNGs generated during the build (`atlases/*/figures/`)
2. **Slice plots** — matplotlib slices of volumetric atlases, with label overlays
3. **Custom yabplot overlays** — plot your own per-region data on the 3-D meshes in `derivatives/yabplot/`

Approaches 1 and 2 work immediately after `uv sync`.  
Approach 3 additionally requires `derivatives/yabplot/` to be populated (run `snbb.build_meshes()` or `datalad get derivatives/`).

In [ ]:
import snbb_atlas_pack as snbb
import nibabel as nib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from pathlib import Path
from IPython.display import Image, display

REPO_ROOT = Path(snbb.__file__).resolve().parent.parent
print("Repo root:", REPO_ROOT)

---

## 1. Pre-built Atlas Figures

The build pipeline generates PNG figures for every atlas via `scripts/visualize_atlases.py`.  
These cover all major viewing angles (lateral, medial, dorsal, ventral, anterior, posterior).

In [ ]:
def show_figure(atlas_id: str, component: str = "subcortical"):
    """Display a pre-built PNG for an atlas."""
    fig_path = REPO_ROOT / "atlases" / f"atlas-{atlas_id}" / "figures" / f"atlas-{atlas_id}_{component}.png"
    if fig_path.exists():
        display(Image(str(fig_path)))
    else:
        print(f"Figure not found: {fig_path}")
        print("Run `snbb.build_meshes()` or `datalad get atlases/` to generate figures.")

In [ ]:
# Tian S1 — subcortical
show_figure("TianS1", "subcortical")

In [ ]:
# HCPex — cortical surface
show_figure("HCPex", "cortical")

In [ ]:
# HCPex — subcortical structures
show_figure("HCPex", "subcortical")

In [ ]:
# HCP-MMP 1.0 — cortical surface (fsLR 32k)
show_figure("HCPMMP", "cortical")

In [ ]:
# Schaefer400 + TianS1 — cortical + subcortical
show_figure("Schaefer2018N400n7Tian2020S1", "cortical")

In [ ]:
show_figure("Schaefer2018N400n7Tian2020S1", "subcortical")

In [ ]:
# Survey all available pre-built figures
available = []
for atlas_id in snbb.list_atlases():
    fig_dir = REPO_ROOT / "atlases" / f"atlas-{atlas_id}" / "figures"
    pngs = sorted(fig_dir.glob("*.png")) if fig_dir.exists() else []
    if pngs:
        available.append({"atlas_id": atlas_id, "figures": ", ".join(p.stem.split("_")[-1] for p in pngs)})

print(f"{len(available)} atlases with pre-built figures:")
for row in available:
    print(f"  {row['atlas_id']:<40s}  [{row['figures']}]")

---

## 2. Slice Plots with matplotlib

For quick volumetric inspection — especially useful for checking registration or region coverage.

In [ ]:
def plot_atlas_slices(atlas_id: str, n_slices: int = 5, axis: int = 1):
    """Plot n_slices equally-spaced slices through a volumetric atlas."""
    atlas = snbb.get_atlas(atlas_id)
    assert atlas.modality == "volumetric", "Surface atlases cannot be sliced this way."

    img = nib.load(atlas.maps)
    data = img.get_fdata()

    n_labels = int(data.max())
    cmap = plt.cm.get_cmap("tab20", n_labels)

    dim = data.shape[axis]
    slice_indices = np.linspace(dim * 0.25, dim * 0.75, n_slices, dtype=int)

    fig, axes = plt.subplots(1, n_slices, figsize=(3 * n_slices, 3.5))
    fig.suptitle(atlas_id, fontsize=13, y=1.01)

    for ax, idx in zip(axes, slice_indices):
        slc = np.take(data, idx, axis=axis).T
        ax.imshow(slc, origin="lower", cmap=cmap, vmin=0, vmax=n_labels, interpolation="nearest")
        ax.set_title(f"slice {idx}", fontsize=8)
        ax.axis("off")

    plt.tight_layout()
    plt.show()

In [ ]:
# Tian S1 — coronal slices
plot_atlas_slices("TianS1", n_slices=5, axis=1)

In [ ]:
# Tian S4 — axial slices (finer subcortical detail)
plot_atlas_slices("TianS4", n_slices=5, axis=2)

In [ ]:
# HCPex — axial slices show both cortex and subcortex
plot_atlas_slices("HCPex", n_slices=6, axis=2)

In [ ]:
# Schaefer400+TianS2 — coronal through the combined atlas
plot_atlas_slices("Schaefer2018N400n7Tian2020S2", n_slices=5, axis=1)

In [ ]:
# Subcortical zoom: extract a tight bounding box around non-zero voxels
atlas = snbb.get_atlas("TianS2")
data = nib.load(atlas.maps).get_fdata()

# Bounding box of non-zero voxels
nz = np.argwhere(data > 0)
lo, hi = nz.min(axis=0), nz.max(axis=0)
crop = data[lo[0]:hi[0]+1, lo[1]:hi[1]+1, lo[2]:hi[2]+1]

n_labels = int(crop.max())
cmap = plt.cm.get_cmap("tab20", n_labels)

mid = [s // 2 for s in crop.shape]
views = [("Axial", crop[:, :, mid[2]].T), ("Coronal", crop[:, mid[1], :].T), ("Sagittal", crop[mid[0], :, :].T)]

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
fig.suptitle("TianS2 — subcortical zoom", fontsize=12)
for ax, (title, slc) in zip(axes, views):
    ax.imshow(slc, origin="lower", cmap=cmap, vmin=0, vmax=n_labels, interpolation="nearest")
    ax.set_title(title)
    ax.axis("off")
plt.tight_layout()
plt.show()

---

## 3. Custom yabplot Overlays

The prebuilt mesh directories in `derivatives/yabplot/` let you overlay **your own per-region values** (e.g. FA, CBF, connectivity strength) on the 3-D atlas geometry — without rebuilding any meshes.

Use `snbb.get_mesh(atlas_id)` to locate the right directory, and `snbb.list_meshes()` to see what is already available.

In [ ]:
import yabplot as yab

# Check what mesh directories are already built
built = snbb.list_meshes()
if built:
    print("Built mesh components:")
    for atlas_id, components in built.items():
        print(f"  {atlas_id:<45s} {components}")
else:
    print("No meshes built yet. Run:\n  snbb.build_meshes()\nor:\n  datalad get derivatives/yabplot/")

In [ ]:
# Build meshes on demand if not yet available
# (This calls visualize_atlases.py internally — skip if already built)

if "TianS1" not in built:
    print("Building TianS1 mesh (this may take a few minutes)…")
    snbb.build_meshes("TianS1")
    print("Done.")
else:
    print("TianS1 mesh already built — skipping.")

### 3a. Subcortical overlay — Tian S1

In [ ]:
atlas = snbb.get_atlas("TianS1")
mesh_dir = snbb.get_mesh("TianS1", component="subcortical")

# Simulate per-region data (replace with your real values, e.g. from a CSV)
rng = np.random.default_rng(42)
data_values = rng.uniform(0, 1, size=len(atlas.labels))

print(f"Atlas: {atlas.atlas_id}  |  {len(atlas.labels)} regions")
print(f"Mesh dir: {mesh_dir}")
print(f"Data shape: {data_values.shape}")
print(f"  Region order matches TSV index column (1-based)")
atlas.labels[["index", "label", "hemisphere"]].assign(value=data_values)

In [ ]:
# Render to PNG (off-screen)
out_path = "/tmp/tians1_custom_overlay.png"

yab.plot_subcortical(
    data=data_values,
    custom_atlas_path=str(mesh_dir),
    export_path=out_path,
    display_type="none",
    figsize=(2000, 700),
)

display(Image(out_path))

### 3b. Subcortical overlay — Tian S4 (fine-grained)

In [ ]:
if "TianS4" not in built:
    snbb.build_meshes("TianS4")

atlas_s4 = snbb.get_atlas("TianS4")
mesh_s4 = snbb.get_mesh("TianS4", component="subcortical")

# Gradient along the left–right axis (x_cog)
x_vals = atlas_s4.labels["x_cog"].values
data_s4 = (x_vals - x_vals.min()) / (x_vals.max() - x_vals.min())

out_s4 = "/tmp/tians4_lateral_gradient.png"
yab.plot_subcortical(
    data=data_s4,
    custom_atlas_path=str(mesh_s4),
    export_path=out_s4,
    display_type="none",
    figsize=(2000, 700),
)
display(Image(out_s4))

### 3c. Cortical overlay — HCPex

In [ ]:
if "HCPex" not in built or "cortical" not in built.get("HCPex", []):
    snbb.build_meshes("HCPex")

hcpex = snbb.get_atlas("HCPex")
mesh_hcpex = snbb.get_mesh("HCPex", component="cortical")

# Colour by lobe membership: assign a numeric code to each lobe
lobe_codes = {lobe: i for i, lobe in enumerate(hcpex.labels["lobe"].dropna().unique())}
data_lobe = hcpex.labels["lobe"].map(lobe_codes).fillna(-1).values.astype(float)

out_hcpex = "/tmp/hcpex_lobe_cortical.png"
yab.plot_cortical(
    data=data_lobe,
    custom_atlas_path=str(mesh_hcpex),
    export_path=out_hcpex,
    display_type="none",
    figsize=(2000, 800),
)
display(Image(out_hcpex))

### 3d. Schaefer+Tian — shared subcortical mesh

All 40 Schaefer+TianS{N} atlases reuse the prebuilt TianS{N} subcortical mesh.  
`get_mesh("Schaefer2018N400n7Tian2020S2", "subcortical")` returns the same directory as `get_mesh("TianS2")` — no extra build step required.

In [ ]:
schaefer_atlas = snbb.get_atlas("Schaefer2018N400n7Tian2020S2")
schaefer_sub_mesh = snbb.get_mesh("Schaefer2018N400n7Tian2020S2", component="subcortical")
tian_s2_mesh     = snbb.get_mesh("TianS2", component="subcortical")

print(f"Schaefer400+TianS2 subcortical mesh: {schaefer_sub_mesh}")
print(f"TianS2 mesh:                          {tian_s2_mesh}")
print(f"Same directory: {schaefer_sub_mesh == tian_s2_mesh}")

# Subcortical rows only — index order matches TianS2
subcortical_rows = schaefer_atlas.labels[schaefer_atlas.labels["component"] == "subcortex"]
print(f"\nSubcortical regions in Schaefer400+TianS2: {len(subcortical_rows)}")

# Plot subcortical portion with random data
data_sub = np.random.default_rng(7).uniform(0, 1, len(subcortical_rows))
out_schaefer_sub = "/tmp/schaefer400_tians2_subcortical.png"

yab.plot_subcortical(
    data=data_sub,
    custom_atlas_path=str(schaefer_sub_mesh),
    export_path=out_schaefer_sub,
    display_type="none",
    figsize=(2000, 700),
)
display(Image(out_schaefer_sub))

### 3e. Cortical overlay — Schaefer400 (7-network colouring)

In [ ]:
schaefer_cort_mesh = snbb.get_mesh("Schaefer2018N400n7Tian2020S2", component="cortical")

# Network codes for the 7-network solution
NETWORK_ORDER = ["Vis", "SomMot", "DorsAttn", "SalVentAttn", "Limbic", "Cont", "Default"]
net_to_code = {n: i for i, n in enumerate(NETWORK_ORDER)}

cortex_rows = schaefer_atlas.labels[schaefer_atlas.labels["component"] == "cortex"]
data_net = cortex_rows["network"].map(net_to_code).values.astype(float)

out_schaefer_cort = "/tmp/schaefer400_network.png"
yab.plot_cortical(
    data=data_net,
    custom_atlas_path=str(schaefer_cort_mesh),
    export_path=out_schaefer_cort,
    display_type="none",
    figsize=(2000, 800),
)
display(Image(out_schaefer_cort))

---

## 4. Label-coloured Slice Overlay

A slightly richer slice view: colour voxels by region and annotate with region names.

In [ ]:
def plot_labelled_slice(atlas_id: str, axis: int = 2, slice_frac: float = 0.5):
    """Coronal / axial / sagittal slice with named region patches in the legend."""
    atlas = snbb.get_atlas(atlas_id)
    img = nib.load(atlas.maps)
    data = img.get_fdata()

    dim = data.shape[axis]
    idx = int(dim * slice_frac)
    slc = np.take(data, idx, axis=axis).T

    labels_df = atlas.labels
    n_labels = int(labels_df["index"].max())
    cmap = plt.cm.get_cmap("tab20", n_labels)

    # Only show regions present in this slice
    present = np.unique(slc[slc > 0]).astype(int)
    present_df = labels_df[labels_df["index"].isin(present)]

    fig, ax = plt.subplots(figsize=(6, 5))
    ax.imshow(slc, origin="lower", cmap=cmap, vmin=0, vmax=n_labels, interpolation="nearest")
    ax.set_title(f"{atlas_id} — slice {idx} (axis {axis})", fontsize=10)
    ax.axis("off")

    patches = [
        mpatches.Patch(color=cmap((row["index"] - 1) / n_labels), label=row["label"])
        for _, row in present_df.iterrows()
    ]
    ax.legend(handles=patches, loc="lower left", fontsize=6, ncol=2, framealpha=0.8)
    plt.tight_layout()
    plt.show()


plot_labelled_slice("TianS1", axis=2, slice_frac=0.47)

In [ ]:
plot_labelled_slice("TianS2", axis=1, slice_frac=0.42)

---

## 5. HCP-MMP Vertex Plot

For surface atlases, vertex-level data can be visualised with nilearn's surface plotting.

In [ ]:
from nilearn import plotting as nlplot
import templateflow.api as tflow

hcpmmp = snbb.get_atlas("HCPMMP")

# Load left-hemisphere vertex labels
gifti_lh = nib.load(hcpmmp.maps)
vertex_labels_lh = gifti_lh.darrays[0].data.astype(float)

# Fetch the matching fsLR 32k inflated surface from TemplateFlow
surf_lh = tflow.get("fsLR", density="32k", hemi="L", suffix="inflated")

print(f"Vertex array: {vertex_labels_lh.shape}")
print(f"Surface mesh: {surf_lh}")

In [ ]:
fig = nlplot.plot_surf_stat_map(
    str(surf_lh),
    stat_map=vertex_labels_lh,
    hemi="left",
    view="lateral",
    title="HCP-MMP 1.0 — left hemisphere",
    colorbar=False,
    cmap="tab20",
    vmax=180,
)
plt.show()

In [ ]:
# Medial view
fig = nlplot.plot_surf_stat_map(
    str(surf_lh),
    stat_map=vertex_labels_lh,
    hemi="left",
    view="medial",
    title="HCP-MMP 1.0 — medial wall",
    colorbar=False,
    cmap="tab20",
    vmax=180,
)
plt.show()

---

## Summary

| Approach | Tool | Works without mesh build? |
|----------|------|:--------------------------:|
| Pre-built PNG figures | `IPython.display.Image` | ✓ (if `datalad get atlases/` run) |
| Slice plots | matplotlib | ✓ |
| Label-annotated slices | matplotlib | ✓ |
| Surface vertex plot | nilearn | ✓ |
| 3-D subcortical overlay | yabplot | requires `derivatives/yabplot/` |
| 3-D cortical overlay | yabplot | requires `derivatives/yabplot/` |

To build all meshes in one call:
```python
snbb.build_meshes()   # builds all 46 atlases
```